# Visualize LitePT Validation Predictions

This notebook visualizes LitePT `*_pred.npy` validation outputs for the HL320 flat-point dataset. It does **not** interpret the flattened prediction as an image grid. Instead it uses the original geometry: `velodyne/<frame>.bin` for 3D/XY/XZ views and the source CSV `Cxd/Cyd`, `azimuth/vertical`, `slot/pixel` columns from `bridge_manifest.json`.

In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

plt.rcParams["figure.figsize"] = (16, 7)
plt.rcParams["axes.grid"] = False

# Fine-tune run directory. The two variants cover the dash/underscore names used in scripts.
PROJECT_ROOT = Path("/home/a60116606/git_repo/noise_seg/pipline_v0")
RUN_DIR_CANDIDATES = [
    PROJECT_ROOT / "output" / "HL320-output_sam3_manual_104" / "fine_tune_100",
    PROJECT_ROOT / "output" / "HL320_output_sam3_manual_104" / "fine_tune_100",
]
RUN_DIR = next((path for path in RUN_DIR_CANDIDATES if path.exists()), RUN_DIR_CANDIDATES[0])
RESULT_DIR = RUN_DIR / "experiment" / "result"
TAXONOMY_PATH = RUN_DIR / "taxonomy.json"
RUN_MANIFEST_PATH = RUN_DIR / "run_manifest.json"

# Leave None to read labeler_dir from run_manifest.json. Set manually if the manifest was moved.
LABELER_DIR_OVERRIDE = Path("/home/a60116606/git_repo/point_labeler/HL320_output_sam3_manual_104")
if not LABELER_DIR_OVERRIDE.exists():
    LABELER_DIR_OVERRIDE = None

# "auto" usually works. Use "training" if *_pred.npy contains dense ids 0..N-1.
# Use "source" if *_pred.npy already contains labels.xml/source ids.
PRED_ID_MODE = "auto"

print("RUN_DIR:", RUN_DIR)
print("RESULT_DIR:", RESULT_DIR)
print("TAXONOMY_PATH:", TAXONOMY_PATH)
print("RUN_MANIFEST_PATH:", RUN_MANIFEST_PATH)


In [ ]:
def read_json(path: Path) -> dict:
    if not path.is_file():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


taxonomy = read_json(TAXONOMY_PATH)
run_manifest = read_json(RUN_MANIFEST_PATH)
labeler_dir = LABELER_DIR_OVERRIDE or Path(run_manifest.get("labeler_dir", ""))
if str(labeler_dir) == ".":
    labeler_dir = Path("")
bridge_manifest = read_json(labeler_dir / "bridge_manifest.json") if str(labeler_dir) else {}
frame_manifest_by_id = {str(frame.get("frame_id")): frame for frame in bridge_manifest.get("frames", [])}

class_names = list(taxonomy.get("class_names") or run_manifest.get("taxonomy", {}).get("class_names") or [])
training_to_source = list(
    taxonomy.get("training_id_to_source_id")
    or run_manifest.get("taxonomy", {}).get("training_id_to_source_id")
    or range(len(class_names))
)
output_ignore_index = int(taxonomy.get("output_ignore_index", 255))
ignore_source_ids = {int(value) for value in taxonomy.get("ignore_source_ids", [255])}
source_id_to_name = {
    int(source_id): class_names[training_id]
    for training_id, source_id in enumerate(training_to_source)
    if training_id < len(class_names)
}
for ignore_id in ignore_source_ids | {output_ignore_index, 255}:
    source_id_to_name.setdefault(int(ignore_id), "ignore")

pred_files = sorted(RESULT_DIR.glob("*_pred.npy"))
print(f"classes: {len(class_names)}")
print("labeler_dir:", labeler_dir if str(labeler_dir) else "<not found>")
print("bridge frames:", len(frame_manifest_by_id))
print(f"prediction files: {len(pred_files)}")
for path in pred_files[:10]:
    print(" ", path.name)


In [ ]:
def frame_id_from_pred(path: Path) -> str:
    name = path.stem
    return name[:-5] if name.endswith("_pred") else name


def split_table_row(line: str) -> list[str]:
    if "," in line:
        return [value.strip() for value in line.split(",")]
    return line.replace("\t", " ").split()


def stable_color(label_id: int) -> np.ndarray:
    label_id = int(label_id)
    if label_id == 0:
        return np.array([35, 35, 35], dtype=np.uint8)
    if label_id in {255, output_ignore_index} | ignore_source_ids:
        return np.array([125, 125, 125], dtype=np.uint8)
    rng = np.random.default_rng((label_id * 1009 + 17) & 0xFFFFFFFF)
    return rng.integers(40, 240, size=3, dtype=np.uint8)


def colorize(labels: np.ndarray) -> np.ndarray:
    labels = np.asarray(labels)
    image = np.zeros(labels.shape + (3,), dtype=np.uint8)
    for label_id in np.unique(labels):
        image[labels == label_id] = stable_color(int(label_id))
    return image


def class_name(label_id: int) -> str:
    return source_id_to_name.get(int(label_id), f"id_{int(label_id)}")


def print_distribution(labels: np.ndarray, max_rows: int = 40) -> None:
    flat = np.asarray(labels).reshape(-1)
    values, counts = np.unique(flat, return_counts=True)
    pairs = sorted([(int(v), int(c)) for v, c in zip(values, counts)], key=lambda item: item[1], reverse=True)
    rows = ["| class id | name | points | percent |", "|---:|---|---:|---:|"]
    for label_id, count in pairs[:max_rows]:
        rows.append(f"| {label_id} | {class_name(label_id)} | {count} | {100.0 * count / flat.size:.3f}% |")
    display(Markdown("\n".join(rows)))


def infer_id_mode(raw: np.ndarray) -> str:
    if PRED_ID_MODE != "auto":
        return PRED_ID_MODE
    values = set(int(value) for value in np.unique(raw) if int(value) >= 0)
    non_ignore = values - ignore_source_ids - {255, output_ignore_index}
    if class_names and non_ignore and max(non_ignore) < len(class_names):
        return "training"
    return "source"


def prediction_to_source_ids(raw: np.ndarray, mode: str) -> np.ndarray:
    arr = np.asarray(raw)
    if mode == "source":
        return arr.astype(np.int32, copy=False)
    out = np.full(arr.shape, output_ignore_index, dtype=np.int32)
    for training_id, source_id in enumerate(training_to_source):
        out[arr == training_id] = int(source_id)
    out[arr < 0] = output_ignore_index
    return out


In [ ]:
def csv_path_for_frame(frame_id: str) -> Path | None:
    frame = frame_manifest_by_id.get(frame_id, {})
    raw = frame.get("source_csv")
    if raw and Path(raw).is_file():
        return Path(raw)
    if str(labeler_dir):
        for folder in ("csv", "CSV"):
            candidate = labeler_dir / folder / f"{frame_id}.csv"
            if candidate.is_file():
                return candidate
    return None


def image_path_for_frame(frame_id: str) -> Path | None:
    frame = frame_manifest_by_id.get(frame_id, {})
    for key in ("image", "source_image"):
        raw = frame.get(key)
        if raw and Path(raw).is_file():
            return Path(raw)
    if str(labeler_dir):
        for suffix in (".jpg", ".jpeg", ".png", ".bmp", ".webp"):
            candidate = labeler_dir / "image_2" / f"{frame_id}{suffix}"
            if candidate.is_file():
                return candidate
    return None


def valid_litept_points(points: np.ndarray) -> np.ndarray:
    arr = np.asarray(points)
    if arr.ndim != 2 or arr.shape[1] < 3:
        raise ValueError(f"points must have shape [N,>=3], got {arr.shape}")
    valid = np.isfinite(arr[:, :3]).all(axis=1)
    valid &= ~(
        (np.abs(arr[:, 0]) < 1e-4)
        & (np.abs(arr[:, 1]) < 1e-4)
        & (np.abs(arr[:, 2]) < 1e-4)
    )
    return valid


def load_points_for_frame(frame_id: str) -> np.ndarray | None:
    if not str(labeler_dir):
        return None
    path = labeler_dir / "velodyne" / f"{frame_id}.bin"
    if not path.is_file():
        print(f"No point cloud for {frame_id}: {path}")
        return None
    points = np.fromfile(path, dtype=np.float32)
    if points.size % 4 != 0:
        raise ValueError(f"{path} does not contain float32 XYZI data")
    return points.reshape(-1, 4)


def load_csv_columns(frame_id: str) -> dict[str, np.ndarray]:
    path = csv_path_for_frame(frame_id)
    if path is None:
        return {}
    with path.open("r", encoding="utf-8") as handle:
        header = None
        for line in handle:
            if line.strip():
                header = split_table_row(line.strip())
                break
        if header is None:
            return {}
        column_map = {name.strip().casefold(): index for index, name in enumerate(header)}
        wanted = ["cxd", "cyd", "azimuth", "vertical", "slot", "pixel", "intensity"]
        values = {name: [] for name in wanted if name in column_map}
        max_idx = max(column_map[name] for name in values) if values else -1
        for line in handle:
            if not line.strip():
                continue
            tokens = split_table_row(line.strip())
            if len(tokens) <= max_idx:
                continue
            for name in values:
                try:
                    values[name].append(float(tokens[column_map[name]]))
                except ValueError:
                    values[name].append(np.nan)
    return {name: np.asarray(vals, dtype=np.float64) for name, vals in values.items()}


def align_geometry_to_predictions(
    *,
    frame_id: str,
    labels: np.ndarray,
    points: np.ndarray | None,
    csv_cols: dict[str, np.ndarray],
) -> tuple[np.ndarray | None, dict[str, np.ndarray]]:
    """Match LitePT predictions to geometry.

    finetune_litept.py writes DefaultDataset only for valid_litept_points(points).
    Therefore LitePT validation predictions can be shorter than the original .bin/.csv row count.
    """
    label_count = int(labels.size)
    if points is not None:
        if points.shape[0] == label_count:
            return points, csv_cols
        valid = valid_litept_points(points)
        valid_count = int(np.count_nonzero(valid))
        if valid_count == label_count:
            aligned_csv = {
                name: values[valid]
                for name, values in csv_cols.items()
                if values.shape[0] == points.shape[0]
            }
            print(
                f"Aligned geometry for {frame_id}: prediction has {label_count} valid points, "
                f"raw bin has {points.shape[0]} points, dropped {points.shape[0] - label_count} invalid points."
            )
            return points[valid], aligned_csv
        print(
            f"Point/prediction mismatch for {frame_id}: prediction={label_count}, "
            f"raw bin={points.shape[0]}, valid bin={valid_count}."
        )
        return None, csv_cols

    # If only CSV exists, keep it only when it already matches predictions.
    csv_count = next((values.shape[0] for values in csv_cols.values()), None)
    if csv_count == label_count:
        return None, csv_cols
    if csv_count is not None:
        print(f"CSV/prediction mismatch for {frame_id}: prediction={label_count}, csv rows={csv_count}.")
    return None, {}


def load_image(frame_id: str) -> np.ndarray | None:
    path = image_path_for_frame(frame_id)
    if path is None:
        return None
    from PIL import Image
    return np.asarray(Image.open(path).convert("RGB"), dtype=np.uint8)


In [ ]:
def sample_indices(n: int, max_points: int | None) -> np.ndarray:
    idx = np.arange(n)
    if max_points is not None and n > max_points:
        rng = np.random.default_rng(42)
        idx = np.sort(rng.choice(idx, size=max_points, replace=False))
    return idx


def index_fallback_image(labels: np.ndarray, width: int = 512) -> np.ndarray:
    flat = np.asarray(labels).reshape(-1)
    height = int(math.ceil(flat.size / width))
    padded = np.full(height * width, output_ignore_index, dtype=np.int32)
    padded[: flat.size] = flat.astype(np.int32, copy=False)
    return padded.reshape(height, width)


def plot_prediction(pred_path: Path, *, scatter_size: float = 0.7, max_points: int | None = 300_000) -> None:
    raw = np.load(pred_path, allow_pickle=False)
    mode = infer_id_mode(raw)
    labels = prediction_to_source_ids(raw, mode).reshape(-1)
    frame_id = frame_id_from_pred(pred_path)
    print(f"{pred_path.name}: raw shape={raw.shape}, dtype={raw.dtype}, id_mode={mode}, frame_id={frame_id}")
    print_distribution(labels)

    if raw.ndim == 2:
        image_labels = prediction_to_source_ids(raw, mode)
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))
        ax.imshow(colorize(image_labels), interpolation="nearest")
        ax.set_title(f"{frame_id} 2D prediction")
        ax.axis("off")
        plt.show()
        return

    points = load_points_for_frame(frame_id)
    csv_cols = load_csv_columns(frame_id)
    points, csv_cols = align_geometry_to_predictions(
        frame_id=frame_id,
        labels=labels,
        points=points,
        csv_cols=csv_cols,
    )
    image = load_image(frame_id)
    has_points = points is not None and points.shape[0] == labels.size
    has_camera = {"cxd", "cyd"}.issubset(csv_cols) and csv_cols["cxd"].size == labels.size

    if not has_points and not has_camera:
        print("No geometry found. Showing only a diagnostic index fallback; this is not a real range image.")
        fig, ax = plt.subplots(1, 1, figsize=(16, 5))
        ax.imshow(colorize(index_fallback_image(labels)), interpolation="nearest", aspect="auto")
        ax.set_title("diagnostic view by flattened point index")
        plt.show()
        return

    idx = sample_indices(labels.size, max_points)
    cols = colorize(labels[idx]).reshape(-1, 3).astype(np.float32) / 255.0
    fig, axes = plt.subplots(2, 2, figsize=(22, 14))
    axes = axes.reshape(-1)

    if has_points:
        pts = points[idx]
        axes[0].scatter(pts[:, 0], pts[:, 1], c=cols, s=scatter_size, linewidths=0)
        axes[0].set_title("LiDAR XY top view")
        axes[0].set_xlabel("x")
        axes[0].set_ylabel("y")
        axes[0].axis("equal")

        axes[1].scatter(pts[:, 0], pts[:, 2], c=cols, s=scatter_size, linewidths=0)
        axes[1].set_title("LiDAR XZ side view")
        axes[1].set_xlabel("x")
        axes[1].set_ylabel("z")
        axes[1].axis("equal")
    else:
        axes[0].text(0.5, 0.5, "velodyne bin not found", ha="center", va="center")
        axes[1].text(0.5, 0.5, "velodyne bin not found", ha="center", va="center")

    if has_camera:
        cxd = csv_cols["cxd"][idx]
        cyd = csv_cols["cyd"][idx]
        finite = np.isfinite(cxd) & np.isfinite(cyd)
        if image is not None:
            axes[2].imshow(image)
            axes[2].set_xlim(0, image.shape[1])
            axes[2].set_ylim(image.shape[0], 0)
        else:
            axes[2].invert_yaxis()
        axes[2].scatter(cxd[finite], cyd[finite], c=cols[finite], s=scatter_size, linewidths=0, alpha=0.85)
        axes[2].set_title("Camera projection via CSV Cxd/Cyd")
        axes[2].set_xlabel("Cxd")
        axes[2].set_ylabel("Cyd")
    else:
        axes[2].text(0.5, 0.5, "CSV Cxd/Cyd not found", ha="center", va="center")

    if {"azimuth", "vertical"}.issubset(csv_cols) and csv_cols["azimuth"].size == labels.size:
        x = csv_cols["azimuth"][idx]
        y = csv_cols["vertical"][idx]
        title = "Angular view: azimuth / vertical"
        xlabel, ylabel = "azimuth", "vertical"
    elif {"pixel", "slot"}.issubset(csv_cols) and csv_cols["pixel"].size == labels.size:
        x = csv_cols["pixel"][idx]
        y = csv_cols["slot"][idx]
        title = "Acquisition table view: pixel / slot"
        xlabel, ylabel = "pixel", "slot"
    else:
        x = y = None
    if x is not None:
        finite = np.isfinite(x) & np.isfinite(y)
        axes[3].scatter(x[finite], y[finite], c=cols[finite], s=scatter_size, linewidths=0)
        axes[3].set_title(title)
        axes[3].set_xlabel(xlabel)
        axes[3].set_ylabel(ylabel)
        axes[3].invert_yaxis()
    else:
        axes[3].text(0.5, 0.5, "No azimuth/vertical or pixel/slot columns", ha="center", va="center")

    plt.suptitle(f"{frame_id} prediction, {labels.size} points", y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
# Pick one file. For your example this selects 000091_pred.npy if it exists.
if not pred_files:
    raise FileNotFoundError(f"No *_pred.npy files found in {RESULT_DIR}")

preferred = RESULT_DIR / "000091_pred.npy"
pred_path = preferred if preferred.is_file() else pred_files[0]
plot_prediction(pred_path)


In [ ]:
# Browse several predictions.
start = 0
count = 5
for path in pred_files[start : start + count]:
    plot_prediction(path, scatter_size=0.45, max_points=150_000)


In [ ]:
def save_camera_projection_png(pred_path: Path, out_dir: Path | None = None, *, scatter_size: float = 0.7) -> Path:
    raw = np.load(pred_path, allow_pickle=False)
    labels = prediction_to_source_ids(raw, infer_id_mode(raw)).reshape(-1)
    frame_id = frame_id_from_pred(pred_path)
    points = load_points_for_frame(frame_id)
    csv_cols = load_csv_columns(frame_id)
    points, csv_cols = align_geometry_to_predictions(
        frame_id=frame_id,
        labels=labels,
        points=points,
        csv_cols=csv_cols,
    )
    if not {"cxd", "cyd"}.issubset(csv_cols) or csv_cols["cxd"].size != labels.size:
        raise ValueError(f"No matching Cxd/Cyd CSV data for {frame_id}")
    image = load_image(frame_id)
    colors = colorize(labels).reshape(-1, 3).astype(np.float32) / 255.0
    cxd = csv_cols["cxd"]
    cyd = csv_cols["cyd"]
    finite = np.isfinite(cxd) & np.isfinite(cyd)
    fig, ax = plt.subplots(1, 1, figsize=(14, 9))
    if image is not None:
        ax.imshow(image)
        ax.set_xlim(0, image.shape[1])
        ax.set_ylim(image.shape[0], 0)
    else:
        ax.invert_yaxis()
    ax.scatter(cxd[finite], cyd[finite], c=colors[finite], s=scatter_size, linewidths=0, alpha=0.85)
    ax.set_title(f"{frame_id} prediction projected to camera")
    ax.axis("off")
    out_dir = out_dir or (RUN_DIR / "prediction_camera_previews")
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{frame_id}_camera_projection.png"
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return out_path


# Uncomment to save camera projection previews for all validation predictions.
# saved = [save_camera_projection_png(path) for path in pred_files]
# print(f"saved {len(saved)} previews to {saved[0].parent if saved else ''}")
